# 02 - Inteligencia de Priorizacao

Objetivo: transformar o ranking causal de fontes em uma regra operacional que escolha os dois melhores telefones por CPF.

A metrica primaria de escolha e entrega. Leitura continua monitorada como metrica de negocio e sera a metrica primaria do A/B.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.preprocessing import StandardScaler

import utils as u

SEED = 42
RANDOM_SEEDS = list(range(20))
HALF_LIFE_GRID = [30, 60, 90, 120, 180, 270, 365]

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Imports OK')


## 1. Dados, aparicoes e split temporal


In [ ]:
df_disparo_raw, df_telefone_raw = u.carregar_dados()

df_disparo = u.filtrar_status_invalidos(df_disparo_raw)
df_telefone = u.filtrar_telefones_fixos(df_telefone_raw)
df_disparo['envio_datahora'] = pd.to_datetime(df_disparo['envio_datahora'])
df_disparo = df_disparo.sort_values('envio_datahora').reset_index(drop=True)

df_aparicoes_brutas = u.explodir_aparicoes(df_telefone)
df_aparicoes_fonte = u.preparar_aparicoes_por_fonte(df_aparicoes_brutas)
df_phone_cpf = u.preparar_telefone_cpf(df_aparicoes_brutas)
df_phone_meta = u.preparar_metadados_telefone(df_telefone, df_aparicoes_fonte)

cutoff_idx = int(len(df_disparo) * 0.8)
cutoff_time = df_disparo.loc[cutoff_idx, 'envio_datahora']
df_train_disparo = df_disparo[df_disparo['envio_datahora'] < cutoff_time].copy()
df_val_disparo = df_disparo[df_disparo['envio_datahora'] >= cutoff_time].copy()

print('Periodo observado:', df_disparo['envio_datahora'].min(), '->', df_disparo['envio_datahora'].max())
print('Cutoff temporal:', cutoff_time)
print('Treino:', len(df_train_disparo))
print('Validacao:', len(df_val_disparo))


## 2. Ranking causal de sistemas aprendido no treino


In [ ]:
def aprender_ranking_sistemas(df_disparo_periodo, df_aparicoes_fonte_base):
    df_join = u.join_disparo_sistema(df_disparo_periodo, df_aparicoes_fonte_base, causal=True)
    metricas = u.calcular_score_sistema(u.calcular_metricas_sistema(df_join))
    return metricas, df_join

metricas_sistema_train, df_train_sistema = aprender_ranking_sistemas(df_train_disparo, df_aparicoes_fonte)

display(metricas_sistema_train[[
    'id_sistema', 'total_disparos', 'sucessos_entrega', 'taxa_entrega',
    'taxa_leitura', 'wilson_lower_entrega', 'score_sistema'
]])

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = metricas_sistema_train.sort_values('score_sistema')
ax.barh(plot_df['id_sistema'].astype(str), plot_df['score_sistema'], color='steelblue', alpha=0.85)
ax.set_xlabel('Score causal da fonte')
ax.set_title('Ranking de fontes aprendido no treino')
plt.tight_layout()
plt.show()


## 3. Base de eventos e grid search do half-life

Aparicoes futuras ou sem data recebem penalizacao maxima via `dias_desde_atualizacao = 9999`.


In [ ]:
def anexar_score_sistema(df_aparicoes_fonte_base, metricas_sistema):
    out = df_aparicoes_fonte_base.merge(
        metricas_sistema[['id_sistema', 'score_sistema']],
        on='id_sistema', how='left'
    )
    fallback = metricas_sistema['score_sistema'].min() if len(metricas_sistema) else 0.0
    out['score_sistema'] = out['score_sistema'].fillna(fallback)
    return out


def montar_eventos(df_disparo_base, df_aparicoes_scored, df_meta, half_life):
    df = (
        df_disparo_base.merge(
            df_aparicoes_scored[['telefone_numero', 'id_sistema', 'registro_data_atualizacao', 'score_sistema']],
            left_on='contato_telefone', right_on='telefone_numero', how='inner'
        )
        .merge(
            df_meta[['telefone_numero', 'telefone_ddd', 'n_proprietarios', 'penalidade_proprietarios', 'n_sistemas_telefone']],
            on='telefone_numero', how='left'
        )
    )
    df = u.adicionar_features_temporais(df, half_life, reference_col='envio_datahora')
    df['score_aparicao_causal'] = df['score_sistema'] * df['decaimento_temporal']

    eventos = df.groupby([
        'id_disparo', 'cpf', 'contato_telefone', 'envio_datahora', 'status_disparo',
        'telefone_ddd', 'n_proprietarios', 'penalidade_proprietarios', 'n_sistemas_telefone'
    ], as_index=False, dropna=False).agg(
        max_score_origem_tempo=('score_aparicao_causal', 'max'),
        melhor_score_sistema=('score_sistema', 'max'),
        melhor_decaimento=('decaimento_temporal', 'max'),
        melhor_dias_atualizacao=('dias_desde_atualizacao', 'min'),
        proporcao_aparicoes_causais=('tem_data_causal', 'mean'),
        qtd_sistemas_candidatos=('id_sistema', 'nunique'),
    )

    eventos['telefone_ddd'] = eventos['telefone_ddd'].fillna(-1)
    eventos['n_proprietarios'] = eventos['n_proprietarios'].fillna(1).clip(lower=1)
    eventos['penalidade_proprietarios'] = eventos['penalidade_proprietarios'].fillna(1.0)
    eventos['n_sistemas_telefone'] = eventos['n_sistemas_telefone'].fillna(0)
    eventos['log_n_sistemas_telefone'] = np.log1p(eventos['n_sistemas_telefone'])
    eventos['y_entrega'] = eventos['status_disparo'].isin(u.STATUS_ENTREGA).astype(int)
    eventos['y_read'] = (eventos['status_disparo'] == 'read').astype(int)
    return eventos


def preparar_matrizes_modelo(df_eventos, cutoff_time_ref):
    df_train = df_eventos[df_eventos['envio_datahora'] < cutoff_time_ref].copy()
    df_val = df_eventos[df_eventos['envio_datahora'] >= cutoff_time_ref].copy()

    prior_ddd, baseline_ddd = u.calcular_prior_suavizado(df_train, 'telefone_ddd', 'y_entrega')
    df_train = df_train.merge(prior_ddd, on='telefone_ddd', how='left').rename(columns={'score_prior': 'score_ddd'})
    df_val = df_val.merge(prior_ddd, on='telefone_ddd', how='left').rename(columns={'score_prior': 'score_ddd'})
    df_train['score_ddd'] = df_train['score_ddd'].fillna(baseline_ddd)
    df_val['score_ddd'] = df_val['score_ddd'].fillna(baseline_ddd)

    feature_cols = [
        'max_score_origem_tempo',
        'melhor_score_sistema',
        'melhor_decaimento',
        'penalidade_proprietarios',
        'score_ddd',
        'log_n_sistemas_telefone',
        'proporcao_aparicoes_causais',
    ]
    return df_train, df_val, feature_cols, prior_ddd, baseline_ddd


df_aparicoes_train_score = anexar_score_sistema(df_aparicoes_fonte, metricas_sistema_train)
resultados_grid = []

for half_life in HALF_LIFE_GRID:
    df_eventos_hl = montar_eventos(df_disparo, df_aparicoes_train_score, df_phone_meta, half_life)
    df_train_modelo, df_val_modelo, feature_cols, prior_ddd_hl, baseline_ddd_hl = preparar_matrizes_modelo(df_eventos_hl, cutoff_time)

    X_train = df_train_modelo[feature_cols].fillna(0)
    X_val = df_val_modelo[feature_cols].fillna(0)
    y_train = df_train_modelo['y_entrega']
    y_val = df_val_modelo['y_entrega']

    scaler_hl = StandardScaler()
    X_train_s = scaler_hl.fit_transform(X_train)
    X_val_s = scaler_hl.transform(X_val)

    modelo_hl = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED)
    modelo_hl.fit(X_train_s, y_train)
    y_proba_val = modelo_hl.predict_proba(X_val_s)[:, 1]

    resultados_grid.append({
        'half_life': half_life,
        'auc_entrega': roc_auc_score(y_val, y_proba_val),
        'log_loss_entrega': log_loss(y_val, y_proba_val),
        'brier_entrega': brier_score_loss(y_val, y_proba_val),
    })

df_grid = pd.DataFrame(resultados_grid)
df_grid['rank_auc'] = df_grid['auc_entrega'].rank(ascending=False)
df_grid['rank_log_loss'] = df_grid['log_loss_entrega'].rank(ascending=True)
df_grid['rank_brier'] = df_grid['brier_entrega'].rank(ascending=True)
df_grid['rank_medio'] = df_grid[['rank_auc', 'rank_log_loss', 'rank_brier']].mean(axis=1)
BEST_HALF_LIFE = int(df_grid.loc[df_grid['rank_medio'].idxmin(), 'half_life'])

display(df_grid.sort_values('rank_medio'))
print('Half-life escolhido:', BEST_HALF_LIFE)


## 4. Modelo de validacao temporal


In [ ]:
df_eventos_best = montar_eventos(df_disparo, df_aparicoes_train_score, df_phone_meta, BEST_HALF_LIFE)
df_train_modelo, df_val_modelo, feature_cols, prior_ddd_train, baseline_ddd_train = preparar_matrizes_modelo(df_eventos_best, cutoff_time)

X_train = df_train_modelo[feature_cols].fillna(0)
X_val = df_val_modelo[feature_cols].fillna(0)
y_train = df_train_modelo['y_entrega']
y_val = df_val_modelo['y_entrega']

scaler_validacao = StandardScaler()
X_train_s = scaler_validacao.fit_transform(X_train)
X_val_s = scaler_validacao.transform(X_val)

modelo_validacao = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED)
modelo_validacao.fit(X_train_s, y_train)
y_proba_val = modelo_validacao.predict_proba(X_val_s)[:, 1]
df_val_modelo['score_modelo_validacao'] = y_proba_val

metricas_validacao = {
    'auc_entrega': roc_auc_score(df_val_modelo['y_entrega'], y_proba_val),
    'log_loss_entrega': log_loss(df_val_modelo['y_entrega'], y_proba_val),
    'brier_entrega': brier_score_loss(df_val_modelo['y_entrega'], y_proba_val),
    'taxa_entrega_validacao': df_val_modelo['y_entrega'].mean(),
    'taxa_read_validacao': df_val_modelo['y_read'].mean(),
    'taxa_read_top_decile': df_val_modelo.sort_values('score_modelo_validacao', ascending=False).head(max(1, int(len(df_val_modelo) * 0.10)))['y_read'].mean(),
}

for chave, valor in metricas_validacao.items():
    print(f'{chave}: {valor:.4f}')

coef_df = pd.DataFrame({'feature': feature_cols, 'coef_padronizado': modelo_validacao.coef_[0]}).sort_values('coef_padronizado', ascending=False)
display(coef_df)


## 5. Simulacao offline de escolha dos 2 melhores

A regra champion nunca escolhe `random`. Se o modelo nao superar os baselines deterministicos, vence o melhor metodo deterministico observado no holdout.


In [ ]:
def score_phones_at_reference(df_aparicoes_scored, df_meta, reference_time, half_life, prior_ddd, baseline_ddd, scaler, modelo, feature_cols):
    df = df_aparicoes_scored.copy()
    df['reference_time'] = reference_time
    df = u.adicionar_features_temporais(df, half_life, reference_col='reference_time')
    df['score_aparicao_causal'] = df['score_sistema'] * df['decaimento_temporal']

    telefones = df.groupby('telefone_numero', as_index=False).agg(
        max_score_origem_tempo=('score_aparicao_causal', 'max'),
        melhor_score_sistema=('score_sistema', 'max'),
        melhor_decaimento=('decaimento_temporal', 'max'),
        melhor_dias_atualizacao=('dias_desde_atualizacao', 'min'),
        proporcao_aparicoes_causais=('tem_data_causal', 'mean'),
    )
    telefones = telefones.merge(
        df_meta[['telefone_numero', 'telefone_ddd', 'n_proprietarios', 'penalidade_proprietarios', 'n_sistemas_telefone']],
        on='telefone_numero', how='left'
    )
    telefones['telefone_ddd'] = telefones['telefone_ddd'].fillna(-1)
    telefones['n_proprietarios'] = telefones['n_proprietarios'].fillna(1).clip(lower=1)
    telefones['penalidade_proprietarios'] = telefones['penalidade_proprietarios'].fillna(1.0)
    telefones['n_sistemas_telefone'] = telefones['n_sistemas_telefone'].fillna(0)
    telefones['log_n_sistemas_telefone'] = np.log1p(telefones['n_sistemas_telefone'])
    telefones = telefones.merge(prior_ddd.rename(columns={'score_prior': 'score_ddd'}), on='telefone_ddd', how='left')
    telefones['score_ddd'] = telefones['score_ddd'].fillna(baseline_ddd)

    X_score = telefones[feature_cols].fillna(0)
    telefones['score_modelo'] = modelo.predict_proba(scaler.transform(X_score))[:, 1]

    telefones['score_heuristico'] = (
        0.50 * u.normalizar_0_1(telefones['max_score_origem_tempo'])
        + 0.20 * u.normalizar_0_1(telefones['score_ddd'])
        + 0.15 * u.normalizar_0_1(telefones['penalidade_proprietarios'])
        + 0.10 * u.normalizar_0_1(telefones['log_n_sistemas_telefone'])
        + 0.05 * u.normalizar_0_1(telefones['proporcao_aparicoes_causais'])
    )
    return telefones


def selecionar_top2(df_base, metodo, sort_cols, ascending):
    selecionados = (
        df_base.sort_values(['cpf'] + sort_cols, ascending=[True] + ascending)
        .groupby('cpf')
        .head(2)
        .copy()
    )
    selecionados['metodo'] = metodo
    selecionados['rank'] = selecionados.groupby('cpf').cumcount() + 1
    return selecionados


def gerar_selecoes(df_candidatos, incluir_random=True):
    selecoes = [
        selecionar_top2(df_candidatos, 'modelo', ['score_modelo', 'score_heuristico', 'melhor_dias_atualizacao', 'telefone_numero'], [False, False, True, True]),
        selecionar_top2(df_candidatos, 'heuristica', ['score_heuristico', 'melhor_dias_atualizacao', 'telefone_numero'], [False, True, True]),
        selecionar_top2(df_candidatos, 'mais_recente', ['melhor_dias_atualizacao', 'score_heuristico', 'telefone_numero'], [True, False, True]),
        selecionar_top2(df_candidatos, 'alfabetico', ['telefone_numero'], [True]),
    ]
    if incluir_random:
        for seed in RANDOM_SEEDS:
            rng = np.random.default_rng(seed)
            temp = df_candidatos.copy()
            temp['ordem_randomica'] = rng.random(len(temp))
            selecoes.append(selecionar_top2(temp, f'random_{seed:02d}', ['ordem_randomica'], [True]))
    return pd.concat(selecoes, ignore_index=True)


def avaliar_selecao(df_selecoes, df_holdout):
    aval = df_selecoes.merge(df_holdout, on='telefone_numero', how='left')
    resumo = aval.groupby('metodo').agg(
        cpfs=('cpf', 'nunique'),
        telefones_escolhidos=('telefone_numero', 'count'),
        cobertura_holdout=('total_validacao', lambda s: s.notna().mean()),
        taxa_entrega_media_top2=('taxa_entrega_validacao', 'mean'),
        taxa_read_media_top2=('taxa_read_validacao', 'mean'),
    ).reset_index()
    top1 = aval[aval['rank'] == 1].groupby('metodo')[['taxa_entrega_validacao', 'taxa_read_validacao']].mean().reset_index().rename(columns={
        'taxa_entrega_validacao': 'taxa_entrega_top1',
        'taxa_read_validacao': 'taxa_read_top1',
    })
    return resumo.merge(top1, on='metodo', how='left'), aval


telefones_score_cutoff = score_phones_at_reference(
    df_aparicoes_train_score, df_phone_meta, cutoff_time, BEST_HALF_LIFE,
    prior_ddd_train, baseline_ddd_train, scaler_validacao, modelo_validacao, feature_cols
)

df_candidatos_cpf = df_phone_cpf.merge(telefones_score_cutoff, on='telefone_numero', how='inner')
cpfs_com_2 = df_candidatos_cpf.groupby('cpf')['telefone_numero'].nunique().loc[lambda s: s >= 2].index
df_candidatos_cpf = df_candidatos_cpf[df_candidatos_cpf['cpf'].isin(cpfs_com_2)].copy()

holdout_por_telefone = df_val_modelo.groupby('contato_telefone').agg(
    total_validacao=('id_disparo', 'nunique'),
    taxa_entrega_validacao=('y_entrega', 'mean'),
    taxa_read_validacao=('y_read', 'mean'),
).reset_index().rename(columns={'contato_telefone': 'telefone_numero'})

selecoes = gerar_selecoes(df_candidatos_cpf, incluir_random=True)
resumo_metodos, selecoes_avaliadas = avaliar_selecao(selecoes, holdout_por_telefone)

resumo_random = resumo_metodos[resumo_metodos['metodo'].str.startswith('random_')].mean(numeric_only=True).to_frame().T
resumo_random['metodo'] = 'random_media_20_seeds'
resumo_deterministico = resumo_metodos[~resumo_metodos['metodo'].str.startswith('random_')].copy()
resumo_comparacao = pd.concat([resumo_deterministico, resumo_random], ignore_index=True)
resumo_comparacao = resumo_comparacao.sort_values(['taxa_entrega_top1', 'taxa_read_top1'], ascending=False)

display(resumo_comparacao)

baselines_deterministicos = ['heuristica', 'mais_recente', 'alfabetico']
modelo_entrega = resumo_deterministico.loc[resumo_deterministico['metodo'] == 'modelo', 'taxa_entrega_top1'].iloc[0]
melhor_baseline = resumo_deterministico[resumo_deterministico['metodo'].isin(baselines_deterministicos)].sort_values(
    ['taxa_entrega_top1', 'taxa_read_top1'], ascending=False
).iloc[0]

if modelo_entrega > melhor_baseline['taxa_entrega_top1']:
    CHAMPION_METODO = 'modelo'
    CHAMPION_MOTIVO = 'Modelo superou os baselines deterministicos em entrega top1 no holdout.'
else:
    CHAMPION_METODO = melhor_baseline['metodo']
    CHAMPION_MOTIVO = 'Modelo nao superou os baselines deterministicos; escolhido melhor metodo deterministico nao aleatorio.'

print('Champion:', CHAMPION_METODO)
print(CHAMPION_MOTIVO)


## 6. Score operacional final e artefatos


In [ ]:
metricas_sistema_full, df_full_sistema = aprender_ranking_sistemas(df_disparo, df_aparicoes_fonte)
df_aparicoes_full_score = anexar_score_sistema(df_aparicoes_fonte, metricas_sistema_full)

df_eventos_full = montar_eventos(df_disparo, df_aparicoes_full_score, df_phone_meta, BEST_HALF_LIFE)
prior_ddd_full, baseline_ddd_full = u.calcular_prior_suavizado(df_eventos_full, 'telefone_ddd', 'y_entrega')
df_eventos_full_model = df_eventos_full.merge(prior_ddd_full, on='telefone_ddd', how='left').rename(columns={'score_prior': 'score_ddd'})
df_eventos_full_model['score_ddd'] = df_eventos_full_model['score_ddd'].fillna(baseline_ddd_full)

X_full = df_eventos_full_model[feature_cols].fillna(0)
y_full = df_eventos_full_model['y_entrega']
scaler_final = StandardScaler()
X_full_s = scaler_final.fit_transform(X_full)
modelo_final = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED)
modelo_final.fit(X_full_s, y_full)

reference_time_final = df_disparo['envio_datahora'].max()
telefones_com_score = score_phones_at_reference(
    df_aparicoes_full_score, df_phone_meta, reference_time_final, BEST_HALF_LIFE,
    prior_ddd_full, baseline_ddd_full, scaler_final, modelo_final, feature_cols
)

if CHAMPION_METODO == 'modelo':
    sort_cols = ['score_modelo', 'score_heuristico', 'melhor_dias_atualizacao', 'telefone_numero']
    ascending = [False, False, True, True]
elif CHAMPION_METODO == 'heuristica':
    sort_cols = ['score_heuristico', 'melhor_dias_atualizacao', 'telefone_numero']
    ascending = [False, True, True]
elif CHAMPION_METODO == 'mais_recente':
    sort_cols = ['melhor_dias_atualizacao', 'score_heuristico', 'telefone_numero']
    ascending = [True, False, True]
else:
    sort_cols = ['telefone_numero']
    ascending = [True]

df_candidatos_final = df_phone_cpf.merge(telefones_com_score, on='telefone_numero', how='inner')
qtd_telefones_cpf = df_candidatos_final.groupby('cpf')['telefone_numero'].nunique()
cpfs_final_2mais = qtd_telefones_cpf[qtd_telefones_cpf >= 2].index
df_candidatos_final_2mais = df_candidatos_final[df_candidatos_final['cpf'].isin(cpfs_final_2mais)].copy()

melhores_por_cpf = selecionar_top2(df_candidatos_final_2mais, CHAMPION_METODO, sort_cols, ascending)
resultado_escolha = melhores_por_cpf[['cpf', 'telefone_numero', 'score_modelo', 'score_heuristico', 'rank']].pivot(
    index='cpf', columns='rank', values=['telefone_numero', 'score_modelo', 'score_heuristico']
)
resultado_escolha.columns = [f'{col[0]}_{col[1]}' for col in resultado_escolha.columns]
resultado_escolha = resultado_escolha.reset_index().rename(columns={
    'telefone_numero_1': 'telefone_1',
    'telefone_numero_2': 'telefone_2',
    'score_modelo_1': 'score_modelo_1',
    'score_modelo_2': 'score_modelo_2',
    'score_heuristico_1': 'score_heuristico_1',
    'score_heuristico_2': 'score_heuristico_2',
})

assert resultado_escolha['telefone_1'].notna().all()
assert resultado_escolha['telefone_2'].notna().all()

resumo_operacional = {
    'champion_metodo': CHAMPION_METODO,
    'champion_motivo': CHAMPION_MOTIVO,
    'total_cpfs_candidatos': int(qtd_telefones_cpf.shape[0]),
    'cpfs_com_1_telefone': int((qtd_telefones_cpf == 1).sum()),
    'cpfs_com_2_ou_mais_telefones': int((qtd_telefones_cpf >= 2).sum()),
    'cpfs_no_resultado_escolha': int(resultado_escolha['cpf'].nunique()),
    'telefone_2_nulos': int(resultado_escolha['telefone_2'].isna().sum()),
}

resumo_modelo_priorizacao = {
    'cutoff_validacao': cutoff_time,
    'half_life_final': BEST_HALF_LIFE,
    'feature_cols': feature_cols,
    'metricas_validacao_modelo': metricas_validacao,
    'comparacao_offline': resumo_comparacao,
    'champion_metodo': CHAMPION_METODO,
    'champion_motivo': CHAMPION_MOTIVO,
}

u.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
artefatos = {
    'resultado_escolha': resultado_escolha,
    'telefones_com_score': telefones_com_score,
    'metricas_sistema_full': metricas_sistema_full,
    'resumo_modelo_priorizacao': resumo_modelo_priorizacao,
    'resumo_operacional': resumo_operacional,
}
if CHAMPION_METODO == 'modelo':
    artefatos['modelo_logistico'] = modelo_final
    artefatos['scaler'] = scaler_final

for nome, obj in artefatos.items():
    caminho = u.PROCESSED_DIR / f'{nome}.pkl'
    with open(caminho, 'wb') as f:
        pickle.dump(obj, f)
    print(f'{nome} salvo em {caminho}')

print('\nResumo operacional:')
for chave, valor in resumo_operacional.items():
    print(f'{chave}: {valor}')

display(resultado_escolha.head(10))
